# REDCap Survey Generator for Healthcare Indicators

## Objetivo

Gerar um arquivo de importação para o REDCap a partir de uma planilha contendo indicadores assistenciais.

O notebook:

- Gera uma Variável para cada indicador;
- Insere questão sobre cumprimento da meta;
- Insere questão para plano de ação;
- Variáveis são utilizadas no Branching Logic do REDCap.

---

## Contexto

Durante minha atuação na área de Inteligência de Dados em Saúde, uma survey precisava ser criada no REDCap contendo centenas de indicadores.

A criação manual demandaria várias horas de trabalho e aumentaria significativamente o risco de erros.

Este projeto automatiza toda essa geração.

---

## Tecnologias

- Python
- Pandas
- Regex
- Excel
- REDCap

---

## Observação

Os arquivos utilizados originalmente pertencem ao ambiente corporativo e foram removidos por questões de confidencialidade.

Este notebook preserva toda a lógica da solução.

In [ ]:
from pathlib import Path
import re

import pandas as pd

In [ ]:
# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

# Arquivo de entrada (substituído por exemplo genérico)
ARQUIVO_ENTRADA = Path("dados/indicadores.xlsx")

# Pasta onde será salvo o arquivo gerado
PASTA_SAIDA = Path("output")

# Nome da coluna que contém os indicadores
COLUNA_INDICADORES = "Indicadores de Volume"

# Nome do arquivo gerado
ARQUIVO_SAIDA = "redcap_upload.xlsx"

In [ ]:
# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================
def gerar_variavel(nome: str) -> str:
    """
    Converte o nome de um indicador em uma variável compatível com o REDCap.

    Exemplo
    --------
    Mortalidade Hospitalar (%)

    torna-se

    mortalidade_hospitalar
    """

    nome = re.sub(r"[^A-Za-z0-9\s]", "", nome)

    return "_".join(nome.lower().split())


def criar_perguntas(indicador: str, variavel: str) -> list:
    """
    Cria as três perguntas referentes a um indicador.

    Cada indicador gera:

    - pergunta principal;
    - pergunta de verificação da meta;
    - pergunta para plano de ação.
    """

    return [

        {
            "Pergunta": indicador,
            "Variável": variavel
        },

        {
            "Pergunta": "O indicador está fora da meta?",
            "Variável": f"meta_{variavel}"
        },

        {
            "Pergunta": "Descreva o plano de ação para este indicador.",
            "Variável": f"plano_{variavel}"
        }

    ]


def gerar_planilha_redcap(
    dataframe: pd.DataFrame,
    coluna_indicadores: str
) -> pd.DataFrame:
    """
    Gera a planilha de importação do REDCap.

    Parameters
    ----------
    dataframe : DataFrame
        Planilha contendo os indicadores.

    coluna_indicadores : str
        Nome da coluna que contém os indicadores.

    Returns
    -------
    DataFrame
        Estrutura pronta para importação no REDCap.
    """

    if coluna_indicadores not in dataframe.columns:
        raise ValueError(
            f"A coluna '{coluna_indicadores}' não foi encontrada."
        )

    linhas = []

    variaveis_existentes = {}

    # Percorre todos os indicadores da planilha
    for indicador in dataframe[coluna_indicadores]:

        if pd.isna(indicador):
            continue

        indicador = str(indicador).strip()

        variavel = gerar_variavel(indicador)

        # Evita variáveis duplicadas.
        #
        # Exemplo:
        #
        # mortalidade
        # mortalidade
        #
        # torna-se
        #
        # mortalidade
        # mortalidade_2

        if variavel in variaveis_existentes:

            variaveis_existentes[variavel] += 1

            variavel = (
                f"{variavel}_"
                f"{variaveis_existentes[variavel]}"
            )

        else:

            variaveis_existentes[variavel] = 1

        linhas.extend(
            criar_perguntas(
                indicador,
                variavel
            )
        )

    return pd.DataFrame(linhas)


def salvar_planilha(
    dataframe: pd.DataFrame,
    pasta_saida: Path,
    nome_arquivo: str
) -> None:
    """
    Salva o arquivo Excel na pasta especificada.
    """

    pasta_saida.mkdir(exist_ok=True)

    caminho_saida = pasta_saida / nome_arquivo

    dataframe.to_excel(
        caminho_saida,
        index=False
    )

    print("=" * 60)
    print("Arquivo gerado com sucesso!")
    print(f"Local: {caminho_saida.resolve()}")
    print("=" * 60)

In [ ]:
# =============================================================================
# EXECUÇÃO
# =============================================================================

def main():
    """
    Fluxo principal do projeto.
    """

    # -------------------------------------------------------------------------
    # LEITURA DOS DADOS
    # -------------------------------------------------------------------------
    #
    # O arquivo utilizado originalmente foi removido por questões de
    # confidencialidade.
    #
    # Espera-se uma planilha contendo uma coluna semelhante a:
    #
    # Indicadores de Volume
    #
    # Mortalidade Hospitalar
    # NPS
    # Readmissão
    # Tempo Médio de Permanência
    #
    # -------------------------------------------------------------------------

    df = pd.read_excel(ARQUIVO_ENTRADA)

    redcap = gerar_planilha_redcap(
        dataframe=df,
        coluna_indicadores=COLUNA_INDICADORES
    )

    salvar_planilha(
        dataframe=redcap,
        pasta_saida=PASTA_SAIDA,
        nome_arquivo=ARQUIVO_SAIDA
    )


if __name__ == "__main__":
    main()